# <center>Premier League Match Outcome Prediction </center>

This notebook predicts the outcome of English Premier League matches using historical match data.  
The data is scraped from [FBRef](https://fbref.com/) and includes 5 seasons (2018–2022).

---

## Objectives
- Build machine learning models to predict Win/Draw/Loss outcomes.
- Evaluate model accuracy, strengths, and limitations.
- Explore real-world applications (fantasy football, betting, sports analytics).

---

## 1. Imports & Setup

We load the necessary Python libraries for data collection, preprocessing, modeling, and visualization.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.utils.class_weight import compute_class_weight

import keras
from keras.models import Sequential
from keras.layers import Dense, Dropout
import tensorflow as tf

sns.set_style('whitegrid')

## 2. Data collection
We scrape Premier League data (scores, fixtures, and shooting stats) from FBRef for the seasons 2018–2022.  
To avoid hitting rate limits, requests are slowed using `time.sleep()`.  

The output of this step is a combined dataset `matches.csv

In [ ]:
matches = pd.read_csv("matches.csv", index_col=0)
matches.head()
matches.shape


## 3. Data cleaning and preprocessing

3.1 **Checking Dataset Shape**


Each PL season has 38 matches × 20 teams = 760 matches.
Over 5 seasons, 3800 matches are expected.
Our dataset has 3704 rows → missing matches due to scraping the 2022/23 season before completion.

In [ ]:
38*20*5
matches["Team"].value_counts()


**Missing Matches per Team**

For each team over 5 seasons, 190 matches should have been played, however at the time of the data being scraped, the current premier league season of 22/23 is still ongoing and therefore not all matches have been played. 
The missing games from the teams that are currently in the top flight are as follows:

In [ ]:
4+4+4+4+5+5+5+5+5+5+6+6+6+7+4+5+4+4+4+4


This accounts for the entire discrepancy in the amount of rows

3.2 **Checking Matches per round**
 
Here the amount of matches played at each match week is checked, as expected the higher matchweeks (28 - ) have missing matches as they were scraped while the season is still ongoing

In [ ]:
matches["Round"].value_counts()


3.3 **Converting Data Types**
 

Next the datatypes are checked as machine learning algorithms cannot work with non numerical data and the columns that are non numerical will need to be converted if they are to be used to train the algorithm

In [ ]:
matches["Date"] = pd.to_datetime(matches["Date"])
matches.dtypes


3.4 **Encoding Categorical Features**
- Venue → numeric codes (Home/Away)
- Opponent → numeric team codes
- Hour → extracted from match start time
- Day_Code → day of the week

In [ ]:
matches["Venue_Code"] = matches["Venue"].astype("category").cat.codes
matches["Opponent_Code"] = matches["Opponent"].astype("category").cat.codes
matches["Hour"] = matches["Time"].str.replace(":.+","", regex=True).astype("int")
matches["Day_Code"] = matches["Date"].dt.dayofweek
matches.head()


3.5 **Defining the Target Variable**

We predict whether the home team wins (W):



In [ ]:
matches["Target"] = (matches["Result"] == "W").astype("int")


3.6 **Feature Engineering: Form Features**


Rolling mean over the last 5 games to capture team form.



In [ ]:
def form(team, columns, new_columns):
    team = team.sort_values("Date")
    form_stats = team[columns].rolling(5, closed='left').mean()
    team[new_columns] = form_stats
    team = team.dropna(subset=new_columns)
    return team

f_col_names = ["xG", "xGA", "Poss", "Dist", "FK", "PK", "PKatt"]
form_columns = [f"{c}_form" for c in f_col_names]
matches = matches.groupby("Team").apply(lambda x: form(x, f_col_names, form_columns))
matches = matches.droplevel("Team")
matches["Formation_Code"] = matches["Formation"].astype("category").cat.codes
matches.head()


**Advanced Preprocessing: Normalized Team Names & Merged Results**

In [ ]:
class MissingDictionary(dict):
    __missing__ = lambda self, key: key

mapping_vals = {
    "Brighton and Hove Albion": "Brighton",
    "Manchester United": "Manchester Utd",
    "Newcastle United": "Newcastle Utd",
    "Tottenham Hotspur": "Tottenham",
    "West Ham United": "West Ham",
    "Wolverhampton Wanderers": "Wolves",
    "Nottingham Forest": "Nott'ham Forest",
    "West Bromwich Albion": "West Brom",
    "Sheffield United": "Sheffield Utd",
    "Huddersfield Town": "Huddersfield"
}
mapping_ = MissingDictionary(**mapping_vals)


## 4. Model Training
4.1 **Train/Test Split**
Split dataset by date (train until 2022-01-01, test after)


In [ ]:
train = matches[matches["Date"] <= '2022-01-01']
test = matches[matches["Date"] > '2022-01-01']

initial_preds = ["Venue_Code", "Opponent_Code", "Hour", "Day_Code"]
predictors = initial_preds + form_columns + ["Formation_Code"]


4.2 **Random Forest classifier**




In [ ]:
rf_clf = RandomForestClassifier(n_estimators=1000, min_samples_split=10, random_state=42)
rf_clf.fit(train[predictors], train["Target"])
preds_rf = rf_clf.predict(test[predictors])


4.3 **Support Vector Machines**


In [ ]:
# RBF kernel
svm_rbf = SVC(random_state=0)
svm_rbf.fit(train[predictors], train["Target"])
preds_svm_rbf = svm_rbf.predict(test[predictors])

# Polynomial kernel
svm_poly = SVC(kernel='poly', degree=4, C=10, random_state=0)
svm_poly.fit(train[predictors], train["Target"])
preds_svm_poly = svm_poly.predict(test[predictors])


4.4 **Deep Neural Network**

In [ ]:
model = Sequential()
model.add(Dense(32, input_dim=len(predictors), activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model.add(Dense(16, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model.add(Dense(8, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model.add(Dense(1, activation='sigmoid'))

model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(train[predictors], train["Target"], epochs=1000, validation_split=0.2, callbacks=[early_stop])

y_pred_prob = model.predict(test[predictors])
y_pred_dnn = np.round(y_pred_prob)


4.5 **Extra DNN with Dropout**

In [ ]:
model2 = Sequential()
model2.add(Dense(64, input_dim=len(predictors), activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model2.add(Dropout(0.5))
model2.add(Dense(32, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model2.add(Dropout(0.5))
model2.add(Dense(16, activation='relu', kernel_initializer=tf.keras.initializers.GlorotUniform(seed=42)))
model2.add(Dropout(0.5))
model2.add(Dense(1, activation='sigmoid'))

model2.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
history2 = model2.fit(train[predictors], train["Target"], epochs=1000, validation_split=0.2, callbacks=[early_stop])

y_pred_prob2 = model2.predict(test[predictors])
y_pred_dnn_extra = np.round(y_pred_prob2)


## 5. Model Evaluation 

In [ ]:
def Fancy_Class_Report(y_true, y_pred):
    report = classification_report(y_true,y_pred, output_dict=True)
    classes = list(report.keys())[:-3]
    metrics = ['precision', 'recall', 'f1-score', 'support']
    classReport = np.array([[report[c][m] for m in metrics] for c in classes])
    sns.heatmap(classReport, annot=True, fmt='.2f', cmap='Blues', xticklabels=metrics, yticklabels=classes, vmin=0, vmax=1)
    plt.title('Classification Report pt.1')
    plt.show()
    
    macro_avg = [report['macro avg'][m] for m in metrics]
    weighted_avg = [report['weighted avg'][m] for m in metrics]
    ificationReport = np.array([macro_avg, weighted_avg])
    sns.heatmap(ificationReport, annot=True, fmt='.2f', cmap='Greens', xticklabels=metrics, yticklabels=['macro avg','weighted avg'])
    plt.title('Classification Report pt.2')
    plt.show()
    
def Fancy_Confusion_Matrix(y_true, y_pred):
    conf_matrix = confusion_matrix(y_true, y_pred)
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Reds', xticklabels=['Actually W','Actually D/L'], yticklabels=['Predicted W','Predicted D/L'])
    plt.title('Confusion Matrix')
    plt.show()


5.1 Evaluate Random Forest


In [ ]:
Fancy_Class_Report(test["Target"], preds_rf)
Fancy_Confusion_Matrix(test["Target"], preds_rf)



In [ ]:
accuracyRF = accuracy_score(test["Target"], preds_rf)
print(f"Accuracy: {accuracyRF}")


In [ ]:
combRF = pd.DataFrame(dict(
    Actual=test["Target"],
    Predicted=preds_rf
))
print(combRF)

5.2 Evaluate SVM (RBF)


In [ ]:

Fancy_Class_Report(test["Target"], preds_svm_rbf)
Fancy_Confusion_Matrix(test["Target"], preds_svm_rbf)



In [ ]:
accuracySVMRBF = accuracy_score(test["Target"], preds_svm_rbf)
print(f"Accuracy: {accuracySVMRBF}")


In [ ]:
combSVMRBF = pd.DataFrame(dict(
    Actual=test["Target"],
    Predicted=preds_svm_rbf
))
print(combSVMRBF)


5.3 Evaluate SVM (poly)

In [ ]:
Fancy_Class_Report(test["Target"], preds_svm_poly)
Fancy_Confusion_Matrix(test["Target"], preds_svm_poly)



In [ ]:
accuracySVMPoly = accuracy_score(test["Target"], preds_svm_poly)
print(f"Accuracy: {accuracySVMPoly}")


In [ ]:
combSVMPoly = pd.DataFrame(dict(
    Actual=test["Target"],
    Predicted=preds_svm_poly
))
print(combSVMPoly)


5.4 Evaluate DNN

In [ ]:

Fancy_Class_Report(test["Target"], y_pred_dnn)
Fancy_Confusion_Matrix(test["Target"], y_pred_dnn)


In [ ]:
accuracydnn = accuracy_score(test["Target"], y_pred_dnn)
print(f"Accuracy: {accuracydnn}")
        

In [ ]:
combdnn = pd.DataFrame(dict(
    Actual=test["Target"],
    Predicted=y_pred_dnn
))
print(combdnn)


5.5 Evaluate DNN Extra Layers

In [ ]:

Fancy_Class_Report(test["Target"], y_pred_dnn_extra)
Fancy_Confusion_Matrix(test["Target"], y_pred_dnn_extra)

In [ ]:
accuracydnnextra = accuracy_score(test["Target"], y_pred_dnn_extra)
print(f"Accuracy: {accuracydnnextra}")


In [ ]:
combdnnextra = pd.DataFrame(dict(
    Actual=test["Target"],
    Predicted=y_pred_dnn_extra
))
print(combdnnextra)


# 6. Performance Comparison

### Classification report comparison: 
| model type | precision | recall | f1-score| support|
| :-: | :-: | :-: | :-:| :-: |
|Random forest classifier| 0.60 | 0.62 | 0.60 | 1052|
|Random forest classifier modified| 0.64 | 0.65 | 0.62 | 1047|
|SVM model with rbf kernel | 0.64 | 0.64 | 0.52 | 1047|
|SVM model with poly kernel |0.64|0.64|0.57|1047|
|deep neural network |0.62|0.63|0.61|1047|
|deep neural network with extra layers|0.62|0.61|0.48|1047|

### Accuracy comparison: 
- Random forest classifier:61.6%
- Random forest classifier modified:64.8%
- SVM model with rbf kernel :63.7%
- SVM model with poly kernel : 63.8%
- deep neural network :63.6%
- deep neural network with extra layers:61.3%

As seen, preprocessing improved the random forest the most, making it the best performing model.


In [ ]:
accs = [ accuracyRF, accuracySVMPoly, accuracydnn, accuracydnnextra]
labels = ['Baseline', 'RF', 'SVM Poly', 'DNN', 'DNN Extra']
plt.bar(labels, accs)
plt.ylabel('Accuracy')
plt.title('Baseline vs Models')
plt.show()


## Conclusion
This case study demonstrates machine learning applications in sports analytics:


**Use Cases:**
- Fantasy football predictions
- Betting odds and strategy analysis
- Sports journalism insights and fan engagement


**Key Takeaways:**
- Proper feature engineering (team form, normalized team names) boosts model accuracy.
- Random Forests are robust for non-linear datasets.
- Hyperparameter tuning (GridSearchCV/RandomizedSearchCV) may be computationally expensive on large datasets.
